# Functions and Imports (no user input)

## Package Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import openpyxl

from scipy import stats
from scipy.stats import linregress
from scipy.interpolate import interp1d

from sklearn import linear_model
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, QuantileTransformer, RobustScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import mean_squared_error, mean_absolute_error

from yellowbrick.cluster import KElbowVisualizer

from datetime import date

import holoviews as hv
import colorcet
# from holoviews.operation import histogram
from bokeh.models import HoverTool
hv.extension('bokeh')

from rdkit import Chem
from rdkit.Chem.Draw import rdMolDraw2D

from IPython.display import SVG


## Determine column name and index during import

In [ ]:
## Allow for user to input both name (string) or index number for column headers
def get_column_loc(column, dataframe):
    if isinstance(column, str):
        return dataframe.columns.get_loc(column), column
    else:
        return column, dataframe.columns[column]

## Image generation from SMILES string

In [ ]:
# adapted from https://birdlet.github.io/2018/06/06/rdkit_svg_web/
def DrawMol(dataframe, smiles_column_loc, image_column, molSize=(200, 100), kekulize=True):
    images = []
    for smiles_string in dataframe.iloc[:, smiles_column_loc]:
        try:
            mc = Chem.MolFromSmiles(smiles_string)
            if kekulize:
                try:
                    Chem.Kekulize(mc)
                except:
                    mc = Chem.Mol(smiles_string.ToBinary())

            if not mc.GetNumConformers():
                Chem.rdDepictor.Compute2DCoords(mc)

            drawer = rdMolDraw2D.MolDraw2DSVG(*molSize)
            drawer.DrawMolecule(mc)
            drawer.FinishDrawing()
            svg = drawer.GetDrawingText().replace('svg:', '')
            images.append(SVG(svg).data)
        except:
            images.append(None)
    
    try:
        dataframe.insert(smiles_column_loc+1, image_column, images)
    except: #  reason for error 
        dataframe[image_column] = images
        
    return dataframe

## HoloViews

In [ ]:
def scatter_plot(
    dataframe: pd.DataFrame, 
    x: str,  # x-axis data
    y: str,  # y-axis data
    title: str = 'default',  # title of plot
    x_label: str = 'default',  # axis label to be printed on plot (does not need to match dataframe name)
    x_range: tuple = None,  # range of x-axis
    y_label: str = 'default',  # axis label to be printed on plot (does not need to match dataframe name)
    y_range: tuple = None,  # range of y-axis
    legend: str = '',  # string with data label if using classifiers/building plots by category
    svgs: str = None,  # string with column name of svgs 
    hover_list: list = None,  # list of column names with data to be shown on hover 
    marker: str = 'o',  # marker type - most of the matplotlib markers are supported (https://matplotlib.org/stable/api/markers_api.html)
    bubbleplot: bool = False,  # if True, will create a bubble plot
    size: int = 10,  # size of markers (recommended: 10-20)
    bubblesize: str = None,  # string with column name for size of points in bubbleplot
    heatmap: bool = False,  # if True, will create a heatmap
    heatmap_col: str = '',  # color of heatmap
    heatmap_label: str = 'default', # label for heatmap colorbar
    heatmap_color: str = 'Plasma',  # color of heatmap
    color: str = '#931319',  # color of markers
    outline: str = '#29323d',  # color of marker outline
    line_width: int = 1,  # width of marker outline
    alpha: int = 1,  # transparency of markers
    groupby: str = None,  # string with column name to group data by
    height: int = 500,  #plot height (recommended: 500)
    width: int = 500,  #plot width (recommended: 500)
    fontscale: int = 1.2,  # scale of font size
):
    
    """
    scatter_plot function based off of HoloViews 'Scatter' element. See documentation for more information:
    hv.help(hv.Scatter)
    https://holoviews.org/reference/elements/bokeh/Scatter.html
    """

    if x_label == 'default':  # if no x_label provided, use x column name
        x_label = x
    if y_label == 'default':  # if no y_label provided, use y column name
        y_label = y
    if heatmap_label == 'default':  # if no heatmap_label provided, use heatmap_col column name
        heatmap_label = heatmap_col

    if not x_range:
        x_min = min(dataframe[x]); x_max = max(dataframe[x])
        x_buffer = abs(x_max-x_min)/10
        x_range = (x_min-x_buffer, x_max+x_buffer)
    if not y_range:
        y_min = min(dataframe[y]); y_max = max(dataframe[y])
        y_buffer = abs(y_max-y_min)/10
        y_range = (y_min-y_buffer, y_max+y_buffer)

    if groupby is not None and hover_list is not None:
        # color = hv.Cycle(color).values
        hover_list.insert(0, groupby)

    if svgs == None and hover_list == None: # no hover information provided
        if title == 'default':  # if no title provided, define from x, y labels
            title = f'{y_label} vs. {x_label}'
        plt = hv.Scatter(dataframe, kdims=[x], vdims=[y], label=legend).opts(title=title, xlabel=x_label, ylabel=y_label, align='center', marker=marker, height=height, width=width, color=color, alpha=alpha, size=size, line_color=outline, line_width=line_width, fontscale=fontscale)
    else:  # hover information provided, build list of hover tools
        hover_list.insert(0, y)
        tooltips = f'<div>end' # beginning of tooltips if no svgs provided
        if svgs != None:
            tooltips = f'<div><div>@{svgs}{{safe}}</div>end'  # beginning of tooltips if svgs are provided
            hover_list.insert(1, svgs)
        if len(hover_list) < 4:
            for label in hover_list:
                if label != svgs and label != y:
                    tooltips = tooltips.replace('end', f'<div><span style="font-size: 17px; font-weight: bold;">@{label}</span></div>end')
        else:
            for label in hover_list:
                if label != svgs and label != y:
                    tooltips = tooltips.replace('end', f'<div><span style="font-size: 12px;">{label}: @{label}</span></div>end')
        
        tooltips = tooltips.replace('end', '</div>')
        hover = HoverTool(tooltips=tooltips)
        if heatmap == False and bubbleplot == False:  # if no heatmap or bubbleplot, build scatter plot  
            if title == 'default':  # if no title provided, define from x, y labels
                title = f'{y_label} vs. {x_label}'          
            plt = hv.Scatter(dataframe, kdims=[x], vdims=hover_list, label=legend).opts(title=title, xlabel=x_label, ylabel=y_label, align='center', marker=marker, height=height, width=width, tools=[hover], color=color, alpha=alpha, size=size, line_color=outline, line_width=line_width, fontscale=fontscale)

        elif heatmap == True and bubbleplot == False:
            if heatmap_col not in hover_list:
                hover_list.append(heatmap_col)
            if title == 'default':  # if no title provided, define from x, y labels
                title = f'{y_label} vs. {x_label}, colored by {heatmap_col}'
            plt = hv.Scatter(dataframe, kdims=[x], vdims=hover_list, label=legend).opts(title=title, xlabel=x_label, ylabel=y_label, align='center', marker=marker, height=height, width=width, tools=[hover], color=heatmap_col, cmap=heatmap_color, colorbar=True, clabel=heatmap_label, alpha=alpha, size=size, line_color=outline, line_width=line_width, fontscale=fontscale)

        elif heatmap == False and bubbleplot == True:
            if bubblesize not in hover_list:
                hover_list.append(bubblesize)
            if title == 'default':  # if no title provided, define from x, y labels
                title = f'{y_label} vs. {x_label}, sized by {bubblesize}'
            min_size = min(dataframe[bubblesize]); max_size = max(dataframe[bubblesize])
            plt = hv.Scatter(dataframe, kdims=[x], vdims=hover_list, label=legend).opts(title=title, xlabel=x_label, ylabel=y_label, align='center', marker=marker, height=height, width=width, tools=[hover], color=color, alpha=alpha, size=((hv.dim(bubblesize)-min_size)/(max_size-min_size)*(max_size-min_size)+min_size)*6*size, line_color=outline, line_width=line_width, fontscale=fontscale)

        elif heatmap == True and bubbleplot == True:
            if heatmap_col not in hover_list:
                hover_list.append(heatmap_col)
            if bubblesize not in hover_list:
                hover_list.append(bubblesize)

            if title == 'default':
                title = f'{y_label} vs. {x_label}, colored by {heatmap_col}, sized by {bubblesize}'
            min_size = min(dataframe[bubblesize]); max_size = max(dataframe[bubblesize])
            plt = hv.Scatter(dataframe, kdims=[x], vdims=hover_list, label=legend).opts(title=title, xlabel=x_label, ylabel=y_label, align='center', marker=marker, height=height, width=width, tools=[hover], color=heatmap_col, cmap=heatmap_color, colorbar=True, clabel=heatmap_label, alpha=alpha, size=((hv.dim(bubblesize)-min_size)/(max_size-min_size)*(max_size-min_size)+min_size)*6*size, line_color=outline, xlim=x_range, ylim=y_range, line_width=line_width, fontscale=fontscale)
        
        if groupby != None:
            # color = hv.Cycle(color).values
            plt = plt.opts(color=groupby, cmap=color)

        return plt
        

In [ ]:
def plot_slope(
    dataframe: pd.DataFrame, 
    x: str,  # string with column name, used to determine slope
    y: str,  # string with column name, used to determine slope
    x_label: str = 'default',  # axis label to be printed on plot (does not need to match dataframe name)
    y_label: str = 'default',  # axis label to be printed on plot (does not need to match dataframe name)
    color: str = '#000000',  # color of slope line
    line_width: int = 2,  # width of slope line
    alpha: int = 1,  # transparency of slope line
    height: int = 500,  #plot height (recommended: 500)
    width: int = 500  #plot width (recommended: 500)
):
    
    
    if x_label == 'default':  # if no x_label provided, use x column name
        x_label = x
    if y_label == 'default':  # if no y_label provided, use y column name
        y_label = y

    slope, intercept, r_value, p_value, std_err = stats.linregress(dataframe[x], dataframe[y])
    slope_plt = hv.Slope(slope, intercept).opts(xlabel=x_label, ylabel=y_label, line_color=color, line_width=line_width, alpha=alpha, height=height, width=width)
    return slope_plt, r_value

In [ ]:
def plot_confidenceinterval(
        dataframe: pd.DataFrame,  # dataframe
        x: str,  # string with column name, used to determine confidence interval
        y: str,  # string with column name, used to determine confidence interval
        x_label: str = 'default',  # axis label to be printed on plot (does not need to match dataframe name)
        x_range: tuple = None,  # range of x-axis
        y_label: str = 'default',  # axis label to be printed on plot (does not need to match dataframe name)
        y_range: tuple = None,  # range of y-axis
        ci: int = 0.999,  # confidence interval (0.9-0.99 recommended)
        color: str = '#5289a1',  # color of confidence interval
        outline: str = '#FFFFFF',  # color of confidence interval line
        alpha: int = 0.2,  # transparency of confidence interval
        height: int = 500,  #plot height (recommended: 500)
        width: int = 500  #plot width (recommended: 500)
):
        
    """ 
    Confidence interval calculations use inferences made on the mean and variance of the distributed data (assumes normal distribution)
    and is calculated by applying a student-t test. Plotting function based off of HoloViews 'Area' element as 'area between curves'. 
    See documentation for more information:
    hv.help(hv.Area)
    https://holoviews.org/reference/elements/bokeh/Area.html
    
    """  

    if x_label == 'default':  # if no x_label provided, use x column name
        x_label = x
    if y_label == 'default':  # if no y_label provided, use y column name
        y_label = y

    if not x_range:
        x_min = min(dataframe[x]); x_max = max(dataframe[x])
        x_buffer = abs(x_max-x_min)/10
        x_range = (x_min-x_buffer, x_max+x_buffer)
    if not y_range:
        y_min = min(dataframe[y]); y_max = max(dataframe[y])
        y_buffer = abs(y_max-y_min)/10
        y_range = (y_min-y_buffer, y_max+y_buffer)

    n = len(dataframe[x])
    t_value = stats.t.ppf(1 - (1 - ci) / 2, n - 2)  # t-value for confidence interval (student-t test for n-2 degrees of freedom)
    x_mean = np.mean(dataframe[x])  # mean of x values
    
    slope, intercept, r_value, p_value, std_err = stats.linregress(dataframe[x], dataframe[y])

    S_xx = (n * np.sum(dataframe[x] ** 2) - np.sum(dataframe[x]) ** 2) / n  # sample-corrected sum of squares (sum of the square of the difference between x and its mean)
    S_xy = (n * np.sum(dataframe[x] * dataframe[y]) - np.sum(dataframe[x]) * np.sum(dataframe[y])) / n  # sample-corrected covariance for x and y 
    S_yy = (n * np.sum(dataframe[y] ** 2) - np.sum(dataframe[y]) ** 2) / n  # sample-corrected sum of squares (sum of the square of the difference between y and its mean)
    
    SSE = S_yy - slope * S_xy # sum of squared estimate of errors (deviation of the observed value from the estimated value)
    s2 = SSE / (n - 2)  #variance of the x, y data
    s = np.sqrt(s2)  # standard deviation of the x, y data

    unique_x = np.unique(dataframe[x])  # unique x values (prevents overplotting of confidence interval)
    mean_upperconfidence_list = slope * unique_x + intercept + t_value * s * np.sqrt((1 / n + (np.square(unique_x - x_mean)) / S_xx))  # line for upper confidence interval
    mean_lowerconfidence_list = slope * unique_x + intercept - t_value * s * np.sqrt((1 / n + (np.square(unique_x - x_mean)) / S_xx))  # line for lower confidence interval

    upper_spread = interp1d(x=unique_x, y=mean_upperconfidence_list, kind='quadratic', fill_value='extrapolate')  # interpolation function for upper confidence interval (smooths line)
    lower_spread = interp1d(x=unique_x, y=mean_lowerconfidence_list, kind='quadratic', fill_value='extrapolate')  # interpolation function for lower confidence interval (smooths line)

    ci_x = np.linspace(min(unique_x) - abs(max(unique_x) - min(unique_x)) / 2, max(unique_x) + abs(max(unique_x) - min(unique_x)) / 2, num=1000)  # x values for confidence interval plot (extends beyond data range)
    ci_upper_y = upper_spread(ci_x)  # y values for upper confidence interval plot corresponding to 'extended' x values
    ci_lower_y = lower_spread(ci_x)  # y values for lower confidence interval plot corresponding to 'extended' x values

    # plot confidence interval
    ci_plt = hv.Area((ci_x, ci_upper_y, ci_lower_y), vdims=['ci_y1', 'ci_y2']).opts(xlabel=x_label, ylabel=y_label, color=color, alpha=alpha, line_color=outline, height=height, width=width, xlim=x_range, ylim=y_range)
    return ci_plt

In [ ]:
def bar_graph(
    dataframe: pd.DataFrame,
    x: str,  # string with column name, used to determine x-axis
    y: str,  # string with column name, used to determine y-axis
    x_label: str = 'default',  # axis label to be printed on plot (does not need to match dataframe name)
    y_label: str = 'default',  # axis label to be printed on plot (does not need to match dataframe name)

    title: str = 'default',  # title of plot
    discrete_x: bool = False,  # if True, will create a bar graph with discrete x-axis
    svgs: str = None,  # string with column name of svgs 
    hover_list: list = None,  # list of column names with data to be shown on hover 
    color: str = '#5289a1',  # color of bars
    alpha: int = 1,  # transparency of bars
    height: int = 500,  #plot height (recommended: 500)
    width: int = 500  #plot width (recommended: 500)
):
    
    """ 
    bar_graph function (if continuous x-axis) based off of HoloViews 'Histogram' element. See documentation for more information:
    hv.help(hv.Histogram)
    http://dev.holoviews.org/reference/elements/bokeh/Histogram.html

    for non-continuous x-axis, bar_graph function is based on 'hv.Bars' element. See documentation for more information:
    hv.help(hv.Bars)
    http://dev.holoviews.org/reference/elements/bokeh/Bars.html
    
    """

    if x_label == 'default':  # if no x_label provided, use x column name
        x_label = x
    if y_label == 'default':  # if no y_label provided, use y column name
        y_label = y
    if title == 'default':  # if no title provided, define from x, y labels
        title = f'{y_label} vs. {x_label}'

    
    if discrete_x == False:  # continuous x-axis, use Histogram element
        if svgs == None and labels == None:
            plt = hv.Histogram(dataframe, kdims=[x], vdims=[y]).opts(xlabel=x_label, ylabel=y_label, title=title, color=color, alpha=alpha, height=height, width=width)
        else: 
            hover_list.insert(0, y)
            tooltips = f'<div>end' # beginning of tooltips if no svgs provided
            if svgs != None:
                tooltips = f'<div><div>@{svgs}{{safe}}</div>end'  # beginning of tooltips if svgs are provided
                hover_list.insert(1, svgs)
            if len(hover_list) < 4:
                for label in hover_list:
                    if label != svgs and label != y:
                        tooltips = tooltips.replace('end', f'<div><span style="font-size: 17px; font-weight: bold;">@{label}</span></div>end')
            else:
                for label in hover_list:
                    if label != svgs and label != y:
                        tooltips = tooltips.replace('end', f'<div><span style="font-size: 12px;">{label}: @{label}</span></div>end')
            
            tooltips = tooltips.replace('end', '</div>')
            hover = HoverTool(tooltips=tooltips)
            plt = hv.Histogram(dataframe, kdims=[x], vdims=hover_list).opts(xlabel=x_label, ylabel=y_label, title=title, tools=[hover], color=color, alpha=alpha, height=height, width=width)
    else:  # discrete x-axis, use Bars element
        if svgs == None and labels == None:
            plt = hv.Bars(dataframe, kdims=[x], vdims=[y]).opts(xlabel=x_label, ylabel=y_label, title=title, color=color, alpha=alpha, height=height, width=width)
        else: 
            hover_list.insert(0, y)
            tooltips = f'<div>end' # beginning of tooltips if no svgs provided
            if svgs != None:
                tooltips = f'<div><div>@{svgs}{{safe}}</div>end'  # beginning of tooltips if svgs are provided
                hover_list.insert(1, svgs)
            if len(hover_list) < 4:
                for label in hover_list:
                    if label != svgs and label != y:
                        tooltips = tooltips.replace('end', f'<div><span style="font-size: 17px; font-weight: bold;">@{label}</span></div>end')
            else:
                for label in hover_list:
                    if label != svgs and label != y:
                        tooltips = tooltips.replace('end', f'<div><span style="font-size: 12px;">{label}: @{label}</span></div>end')
            
            tooltips = tooltips.replace('end', '</div>')
            hover = HoverTool(tooltips=tooltips)
            plt = hv.Bars(dataframe, kdims=[x], vdims=hover_list).opts(xlabel=x_label, ylabel=y_label, title=title, tools=[hover], color=color, alpha=alpha, height=height, width=width, xlim=(min(dataframe[x]), max(dataframe[x])), ylim=(min(dataframe[y]), max(dataframe[y])))
        return plt

## Chemical Space Clustering

### k-Means Clustering

In [ ]:
def kmeans_score(dataframe, k):
    %matplotlib inline
    
    elbow_plot = KElbowVisualizer(KMeans(n_clusters=k, n_init='auto'), random_state=42)
    elbow_plot.fit(dataframe)
    return elbow_plot

def find_kmeans_centroids(dataframe, centroid_coordinates, chemical_space_coordinate_columns, ligand_id_column):

    kmeans_centroids = []
    for i in range(0, len(kmeans_centroid_coordinates)):
        euclidian_distances = np.linalg.norm(dataframe[chemical_space_coordinate_columns].values - kmeans_centroid_coordinates[i], axis=1)
        closest_index = np.argmin(euclidian_distances)
        kmeans_centroids.append(dataframe.iloc[closest_index][id_column])
    # Check:
    if len(kmeans_centroids) != len(kmeans_centroid_coordinates):
        raise ValueError("Number of clusters and number of ligand IDs identified as cluster centroids do not match.")
    return kmeans_centroids


## Sanitize Column Names

In [ ]:
def sanitize_column_names(df):
    df.columns = df.columns.str.replace('[^a-zA-Z0-9]', '_')  # replace non-alphanumeric characters with '_'
    df.columns = df.columns.str.replace('[ ,-]', '_', regex=True)  # replace spaces, commas, and hyphens with '_'
    return df

# Data Import

In [ ]:
file = 'PP_04-21-2025_Reformatted_locked.xlsx'
sheet = 'Sheet1'
header = 0  # row number of header (0-index) (set header = 1 to drop row with x1, x2... column names if present)

id_column = 'Ligand_ID' #name or 0-index
smiles_column = 'SMILES' #name or 0-index (leave blank if not available)
response_column = '' #name or 0-index (leave blank if not available)
descriptor_start_column = 'R_NBO_min' #name or 0-index

# Read in data
if file.endswith('.csv'):
    df = pd.read_csv(file, header=header)
elif file.endswith('.xlsx'):
    df = pd.read_excel(file, sheet, header=header, engine='openpyxl')
else:
    print('File type not supported. Please use .csv, .xlsx, or .pkl file types.')

# # Drop rows with NaN values (will remove smiles column if there are any missing fields)
# df = df.dropna(axis=1, how='any')  

# Generate list of descriptors 
descriptor_start_column_loc, descriptor_start_column = get_column_loc(descriptor_start_column, df)
descriptors = list(df.columns)[descriptor_start_column_loc:]

#  Generate folder for any saved figures, named with run date
run_date = date.today().strftime("%b-%d-%Y")
if not os.path.exists(run_date):
    os.makedirs(run_date)

## Generate Image from SMILES String

Note: RDKit image generation will occasionally produce overlapping groups (primarily for large structures)

In [ ]:
dataframe = df
image_column = 'Image' #name of column that svgs will go in to (not pre-existing)

smiles_column_loc, smiles_column = get_column_loc(smiles_column, dataframe)
df = DrawMol(dataframe, smiles_column_loc, image_column)
svgs = True

# Data Preparation

## Remove Colinear Features

In [ ]:
dataframe = df
threshold = 0.9  # threshold for colinearity (1 = perfect colinearity)

# Calculate correlation matrix
correlation_matrix = dataframe[descriptors].corr().abs()

# Select upper triangle of correlation matrix
upper = correlation_matrix.where(np.triu(np.ones(correlation_matrix.shape), k=1).astype(bool))

# Find index of feature columns with correlation greater than threshold
columns_to_drop = [column for column in upper.columns if any(upper[column] > threshold)]
print(f'{len(descriptors)} descriptors before colinearity cutoff.\n{len(columns_to_drop)} descriptors removed.\n{len(descriptors)-len(columns_to_drop)} remaining.')

# Drop columns with high correlation
dataframe.drop(columns=columns_to_drop, axis=1, inplace=True)

# Regenerate list of descriptors
descriptors = list(dataframe.columns)[descriptor_start_column_loc+1:]

## Feature Scaling

Note: currently gives 'FutureWarning' due to dataframe use, will be updated (error goes away when cell is run twice)

In [ ]:
dataframe = df
scaler = 'Standard'

if scaler == 'Standard':
    scaler = StandardScaler()
if scaler == 'MinMax':
    scaler = MinMaxScaler()
if scaler == 'Quantile':
    scaler = QuantileTransformer(random_state=3)
if scaler == 'Robust':
    scaler = RobustScaler()

# Scale descriptors

if 'Split' in dataframe.columns.to_list():
    dataframe.loc[dataframe['Split'] == 'train', descriptors] = scaler.fit_transform(dataframe.loc[dataframe['Split'] == 'train', descriptors])  # fit and transform the training set
    if 'internal validation' in list(dataframe['Split']):
        dataframe.loc[dataframe['Split'] == 'internal validation', descriptors] = scaler.transform(dataframe.loc[dataframe['Split'] == 'internal validation', descriptors])  # transform the internal validation set
    if 'external validation' in list(dataframe['Split']):
        dataframe.loc[dataframe['Split'] == 'external validation', descriptors] = scaler.transform(dataframe.loc[dataframe['Split'] == 'external validation', descriptors])  # transform the external validation set
else:
    dataframe[descriptors] = scaler.fit_transform(dataframe[descriptors])


# Chemical Space

## PCA

Note: currently produces a warning when features are added to dataframe. Run cell twice to remove warning.

In [ ]:
dataframe = df
n_components = 10  # use n_components = 'mle' for automated solving (uses Minka's MLE), requires more samples than features
# number of principal components to keep (principal components = "axes", recommended to describe ~60-80% of variance)
# number of principal components must be larger than the number of samples or the number of features (whichever is smaller)

# Fit PCA
pca = PCA(n_components=n_components)
pca.fit(dataframe[descriptors])
principal_components = pca.transform(dataframe[descriptors])

# Add principal components to dataframe
if n_components != 'mle':
    for i in range(n_components):
        dataframe[f'pc{i+1}'] = principal_components[:, i]

# Print explained variance
pca_score = pca.explained_variance_ratio_
print(f'{round(np.sum(pca_score)*100, 1)}% of variance explained by {n_components} principal components\n')
print(f'Variance explained by each principal component:')
for i, variance in enumerate(pca_score):
    print(f'pc{i+1}: {round(variance*100, 1)}%')

for index, row in dataframe.iterrows():
    for descriptor in descriptors:
        dataframe.at[index, f'{descriptor}_pc'] = row[descriptor] * pca.components_[0][descriptors.index(descriptor)]
        dataframe.at[index, f'{descriptor}_pc_weight'] = pca.components_[0][descriptors.index(descriptor)]

dataframe.to_csv(f'{run_date}/PCA.csv', index=False)

### View PCA Space

In [ ]:
dataframe = df
x_axis = 'pc1'
y_axis = 'pc2'
hover_list = [id_column]

save_plot = False
file_name = 'pca'
file_path = run_date

plt = scatter_plot(dataframe, x=x_axis, y=y_axis, svgs=image_column, hover_list=hover_list, color='#7291ab', alpha=0.8)

if save_plot:
    hv.save(plt, file_path + '/' + file_name + '.html', fmt='html')

plt

## Cluster PCA Space

In [ ]:
dataframe = df
dimred_columns = [col for col in dataframe.columns if col.startswith('pc')]
k = (4, 12)

clustering_algorithm = 'kMeans'

if clustering_algorithm == 'kMeans':
    elbow_method = kmeans_score(dataframe[dimred_columns], k)
    default_clusters = elbow_method.elbow_value_
    print(f'Optimal number of clusters using distortion score (elbow plot): {default_clusters}')
    elbow_method.show()

dimred_columns = [col for col in dataframe.columns if col.startswith('pc')]

### Add PC Clusters to Dataframe and Identify Centroids

In [ ]:
n_clust = default_clusters # number of clusters, default is defined in cell above (default_clusters)

kmeans_clustering = KMeans(n_clusters=default_clusters, random_state=42, n_init=10)
kmeans_clustering.fit(dataframe[dimred_columns])

dataframe[f'{clustering_algorithm}_cluster'] = kmeans_clustering.labels_ + 1

kmeans_centroid_coordinates = kmeans_clustering.cluster_centers_
kmeans_centroids = find_kmeans_centroids(dataframe, kmeans_centroid_coordinates, chemical_space_coordinate_columns=dimred_columns, ligand_id_column=id_column)

## Plot PC Clusters in PCA Space

In [ ]:
dataframe = df
x_axis = 'pc1'
y_axis = 'pc2'

highlight_centroids = True
cluster_number = 'kMeans_cluster'
centroids = kmeans_centroids

groupby = 'kMeans_cluster'  # name or 0-index for column to group data by
color = 'Category20'  # color palette for grouped data

save_plot = False
file_name = 'Clustered PCA Space'
file_path = run_date

plt = ''
plt = scatter_plot(dataframe, x=x_axis, y=y_axis, title=file_name, svgs=image_column, hover_list=[id_column], groupby=groupby, color=color, outline=color)

if highlight_centroids:
    centroid_plt = scatter_plot(dataframe.loc[dataframe[id_column].isin(centroids)], x=x_axis, y=y_axis, title=file_name, svgs=image_column, hover_list=[id_column, cluster_number], color='#ffffff', outline='#000000', legend='Centroids')
    plt = plt * centroid_plt
    print("Ligand IDs for centroids:")
    print("".join([f"\t{kmeans_centroids[i]} for cluster {i+1}\n" for i in range(len(kmeans_centroids))]))

if save_plot:
    hv.save(plt, file_path + '/' + file_name + '.html', fmt='html')

plt